# 13 · 向量数据库：产品对比与选型

> 向量数据库 = 向量索引 + 元数据过滤 + 持久化 + 高可用。这一课先看“有哪些、怎么选”，下一课再看“内部索引怎么跑”。

**本文件覆盖知识点**：FAISS / Milvus / Qdrant / Weaviate / Chroma / Elasticsearch / OpenSearch / pgvector

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 常见向量数据库对比

| 方案 | 形态 | 适合 | 特点 |
|------|------|------|------|
| **FAISS** | 库(进程内) | 学习/小规模/实验 | 快、轻，无持久化与权限，须自己管理索引文件 |
| **Chroma** | 嵌入式 DB | 本地原型 | 简单、数据落盘 |
| **Milvus / Zilliz** | 分布式服务 | 大规模生产 | 万亿级、自带索引/过滤/高可用 |
| **Qdrant** | 服务 | 中大规模 | Rust 写，过滤强、易用 |
| **Weaviate** | 服务 | 中大规模 | 模块化、混合检索原生 |
| **Elasticsearch / OpenSearch** | 搜索+向量 | 已有 ES 栈 | 关键词与向量一把抓 |
| **pgvector** | Postgres 扩展 | 数据已在 PG | 免新组件，规模中等够用 |

> 选型三角：**数据规模 × 现有技术栈 × 功能需求（过滤/权限/混合检索）**。本课程为便于教学用 FAISS 讲透原理。

In [ ]:
# 知识点·真调说明：向量库选型 —— 给两个真实约束场景做“选型问询”，看规模×栈×功能如何决定答案
print('① 场景A：中小规模 + 已有 Postgres 业务库')
_llm_live(
    prompt='场景A：一家公司已有 Postgres 存业务数据，向量约 300 万条、维度 1024，需要和现有 SQL 数据一起做条件过滤，'
           '不想新增运维组件，团队主要会 SQL。请从 FAISS / pgvector / Chroma / Milvus 中选一个，'
           '说明选择主因，并指出这套方案做到什么规模会开始吃力。',
    system='你是向量数据库选型顾问。要求：先给“推荐：XXX”，再用不超过 3 条理由支撑；不要逐个点评所有产品。',
    fallback='未配置 Key 的固定样例：\n'
             '推荐：pgvector。理由：数据本来就在 Postgres，加扩展即可做向量相似度，能和业务 SQL 共用过滤/事务，'
             '零新增组件、团队不用学新技能。当向量涨到千万级、QPS 高或要独立扩缩容时，pgvector 的索引构建和内存会吃力，'
             '再考虑 Milvus/Qdrant 这类专用服务。',
    temperature=0.2,
)
print()
print('② 场景B：千万到亿级海量 + 多租户权限 + 高可用 + 想兼顾关键词')
_llm_live(
    prompt='场景B：另一家公司做千万到亿级向量的生产检索，多租户数据要隔离、要细粒度权限过滤、要高可用与横向扩展，'
           '还希望关键词与向量检索能统一服务。请先说明“把 FAISS 索引文件直接加载进进程”这种裸 FAISS 直连缺了什么，'
           '再从 Milvus / Qdrant / Elasticsearch 中给出推荐并说明主因。',
    system='你是向量数据库选型顾问。要求：先用 2~3 点说清裸 FAISS 直连的缺口，再给“推荐：XXX”与一句主因，语言精炼。',
    fallback='未配置 Key 的固定样例：\n'
             '裸 FAISS 直连的缺口：它是进程内索引库——无持久化/多副本高可用、无内置元数据权限与租户隔离、'
             '无横向扩缩容与故障转移，全部要自己搭。\n'
             '推荐：若强过滤+易用优先选 Qdrant；若关键词与向量要同一套 API、团队已熟 ES 则选 Elasticsearch。'
             '选型主因是“多租户权限 + 混合检索”往往比离线精度排名更能决定上线成败。',
    temperature=0.2,
)
print()
print('→ 同一个问题没有唯一答案：选型三角“规模×栈×功能”决定取舍，这正是本课表格要训练的判断。')

In [ ]:
# FAISS 最小可用：归一化向量 + 内积索引 = 余弦检索
import numpy as np
import faiss

d = 4  # 演示用低维向量
x = np.array([
    [1, 0, 0, 0],   # doc0
    [0, 1, 0, 0],   # doc1
    [0, 0, 1, 0],   # doc2
], dtype='float32')
index = faiss.IndexFlatIP(d)
index.add(x)

q = np.array([[0.0, 1.0, 0.0, 0.0]], dtype='float32')  # 与 doc1 最像
sims, ids = index.search(q, 2)
print('FAISS 版本:', faiss.__version__)
print('命中 doc:', ids[0], '| 相似度:', sims[0], '（点积=余弦，因已归一化）')
print('\nFAISS 只存向量与编号；文本/元数据要自己用数组/DB 对齐存放。')

## 2. 一个向量库要提供的核心能力

```text
写入:  upsert(id, vector, metadata)
检索:  search(vector, top_k, filter={...})   # 向量相似 + 元数据过滤
管理:  持久化 / 备份 / 高可用 / 一致性
扩展:  分片(scale-out) / 索引类型可配(Flat·IVF·HNSW…)
```

生产选型时：`filter 性能`、`是否支持多租户/权限`、`混合检索(向量+BM25)`、`部署形态` 往往比“谁分高”更重要。



In [ ]:
# 知识点·真调说明：向量库“核心能力” —— 让模型给“FAISS 直连上生产”挑毛病，反推出能力清单
_llm_live(
    prompt='有的团队把检索层写成这样：启动时把 FAISS 索引文件 load 进内存，search 只做相似度，'
           '文本与元数据由自己另开数组/表维护，过滤也自己写。\n'
           '请以生产评审的身份，指出这套直连方案缺了“向量数据库该提供”的哪些能力（分点列出），'
           '最后点出一条最容易被忽视、却往往最早咬人的坑。',
    system='你是资深 RAG 平台架构师。要求分点、每点一句话，最后单列一行给出“最易忽视却最早咬人的一项”。',
    fallback='未配置 Key 的固定样例：\n'
             '- 持久化与故障恢复：索引只落一个文件，进程崩溃或机器故障没有副本与日志，坏了只能全量重建。\n'
             '- 元数据过滤与权限：filter 要自己遍历合并，租户隔离完全缺失。\n'
             '- 高可用与扩缩容：单机内存上限即容量上限，无法水平加节点、无负载均衡。\n'
             '- 一致性：写入/删除与查询之间没有事务语义，更新期间会读到旧数据。\n'
             '最易忽视却最早咬人的，往往是“元数据过滤”：业务几乎都要按租户/时间/类型过滤，'
             '自己实现一复杂就漏过滤或退化成全量扫描。',
    temperature=0.2,
)
print('→ 这正对应本课“写入 / 检索 / 管理 / 扩展”四类核心能力；FAISS 仍是教学最好的载体，生产规模化要换真库。')

## 小结

- FAISS=库，适合学习与实验；生产规模化用 Milvus/Qdrant/ES 等；
- 向量库能力 = 向量检索 + 元数据 + 持久化 + 高可用；
- 选型看**规模×栈×功能**。内部到底怎么“快”，下一课看索引算法。